In [ ]:
!mkdir src

# config

In [ ]:
%%writefile src/config.py

hf_user_name = "amin-oj"
model_checkpoint = "distilbert-base-uncased"
local_artifact_dir = "distilbert_sft_mlm_local"
dataset_dict = {"path": "imdb", "split": "train"}
train_size = 2048
test_size = 512
batch_size = 16
num_train_epochs = 2
gacc_steps = 4
lr=2e-5
lr_scheduler_type = "linear"
chunk_size = 128
ckpt_name = f"{model_checkpoint.split("/")[-1]}-finetuned-ner-{dataset_dict['path']}-accelerate"

# utils

In [ ]:
%%writefile src/utils.py

from kaggle_secrets import UserSecretsClient
import os, torch

def hf_login():
    user_secrets = UserSecretsClient()
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

def get_mp():
    bf16_supported = False
    if torch.cuda.is_available():
        try:
            bf16_supported = torch.cuda.is_bf16_supported()
        except Exception:
            bf16_supported = False

    mixed_precision = "bf16" if bf16_supported else "fp16"
    return mixed_precision

# preprocess

In [ ]:
%%writefile src/preprocess.py

from src.config import chunk_size

def tokenize_function(examples, tokenizer):
    result = tokenizer(examples["text"])
    if tokenizer.is_fast:
        result["word_ids"] = [result.word_ids(i) for i in range(len(result["input_ids"]))]
    return result

def group_texts(examples, chunk_size = chunk_size):
    # Concatenate all texts
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    # Compute length of concatenated texts
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the last chunk if it's smaller than chunk_size
    total_length = (total_length // chunk_size) * chunk_size
    # Split by chunks of max_len
    result = {
        k: [t[i : i + chunk_size] for i in range(0, total_length, chunk_size)]
        for k, t in concatenated_examples.items()
    }
    # Create a new labels column
    result["labels"] = result["input_ids"].copy()
    return result

def insert_random_mask(batch, data_collator):
    features = [dict(zip(batch, t)) for t in zip(*batch.values())]
    masked_inputs = data_collator(features)
    return {"masked_" + k: v.numpy() for k, v in masked_inputs.items()}

# train

## Create dataloaders

- We saw that `DataCollatorForLanguageModeling` also applies random masking with each evaluation, so we’ll see some fluctuations in our perplexity scores with each training run.
- One way to eliminate this source of randomness is to apply the masking _once_ on the whole test set, and then use the default data collator in 🤗 Transformers to collect the batches during evaluation.
- Let’s implement a simple function that applies the masking on a batch
    - We’ll apply this function to our test set and drop the unmasked columns so we can replace them with the masked ones.
    - We can use whole word masking by replacing the `data_collator` above with the appropriate one
        - in which case we shouldn't pop the `word_ids` field from the input batch     
- We can then set up the dataloaders as usual, but we’ll use the `default_data_collator` from 🤗 Transformers for the evaluation set.

In [ ]:
%%writefile src/train.py

from transformers import DataCollatorForLanguageModeling
from torch.utils.data import DataLoader
from transformers import default_data_collator

from src.preprocess import insert_random_mask
from src.config import batch_size

def create_dls(mlm_datasets, tokenizer, mlm_probability=0.15):

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm_probability=mlm_probability
    )
    
    mlm_datasets = mlm_datasets.remove_columns(["word_ids"])
    eval_dataset = mlm_datasets["test"].map(
        insert_random_mask,
        batched=True,
        fn_kwargs = {"data_collator": data_collator},
        remove_columns=mlm_datasets["test"].column_names,
    ).rename_columns(
        {
            "masked_input_ids": "input_ids",
            "masked_attention_mask": "attention_mask",
            "masked_labels": "labels",
        }
    )
    
    train_dataloader = DataLoader(
        mlm_datasets["train"],
        shuffle=True,
        batch_size=batch_size,
        collate_fn=data_collator,
    )
    eval_dataloader = DataLoader(
        eval_dataset,
        batch_size=batch_size,
        collate_fn=default_data_collator
    )
    return train_dataloader, eval_dataloader

### what happens inside the `insert_random_mask` function??

In [ ]:
# test_batch = mlm_datasets["train"][:4]
# test_batch.pop("word_ids")
# feat = [dict(zip(test_batch, t)) for t in zip(*test_batch.values())]
# msk_in = data_collator(feat)
# res = {"masked_" + k: v.numpy() for k, v in msk_in.items()}

# for k,v in test_batch.items():
#     print(f"{k}: {len(v)}")
# print("===============")
# print(len(feat))
# print("===============")
# for i in range(len(feat)):
#     print(feat[i] == mlm_datasets["train"][i])
# print("===============")
# for k,v in res.items():
#     print(f"{k}: {len(v)}")
# print("===============")

# eval

In [ ]:
%%writefile src/evaluation.py

import torch, math
from tqdm.auto import tqdm

def run_eval1(model, eval_dataloader, accelerator):
    model.eval()
    total_loss = torch.tensor(0.0, device=accelerator.device)
    total_count = torch.tensor(0.0, device=accelerator.device)
    pbar = tqdm(
        eval_dataloader,
        desc="Evaluation",
        disable=not accelerator.is_main_process,
        leave=False
    )
    for batch in pbar:
        with torch.no_grad():
            with accelerator.autocast():
                outputs = model(**batch)
                loss = outputs.loss
        bs = batch["labels"].shape[0]
        total_loss += loss.detach() * bs
        total_count += bs

    # Reduce sums across processes => global sums
    total_loss = accelerator.reduce(total_loss, reduction="sum")
    total_count = accelerator.reduce(total_count, reduction="sum")
    eval_loss = (total_loss / torch.clamp(total_count, min=1.0)).item()
    return {
        "eval_loss": eval_loss,
        "perplexity": math.exp(eval_loss)
    }

def run_eval2(model, eval_dataloader, accelerator):
    model.eval()
    losses = []
    pbar = tqdm(
        eval_dataloader,
        desc="Evaluation",
        disable=not accelerator.is_main_process,
        leave=False
    )
    for batch in pbar:
        with torch.no_grad():
            with accelerator.autocast():
                outputs = model(**batch)
        loss = outputs.loss
        bs = batch["labels"].shape[0]
        losses.append(accelerator.gather_for_metrics(loss.repeat(bs)))

    losses = torch.cat(losses)
    eval_loss = torch.mean(losses)
    return {
        "loss_length": losses.shape[0],
        "eval_loss": eval_loss,
        "perplexity": math.exp(eval_loss)
    }

# Main

In [ ]:
%%writefile main.py

# Load deps
import os, torch, math
from tqdm.auto import tqdm
from accelerate import Accelerator
from huggingface_hub import HfApi, create_repo
from datasets import load_dataset
from torch.optim import AdamW
from transformers import AutoTokenizer
from transformers import AutoModelForMaskedLM
from transformers import get_scheduler


from src.config import *
from src.utils import hf_login, get_mp
from src.preprocess import tokenize_function, group_texts
from src.train import create_dls
from src.evaluation import run_eval1, run_eval2



if __name__ == "__main__":
    # Init accelerator
    accelerator = Accelerator(
        gradient_accumulation_steps=gacc_steps,
        mixed_precision=get_mp()
    )
    
    if accelerator.is_main_process:
        print(f"checkpoint name: {ckpt_name}")
        print(f"# GPUs: {accelerator.num_processes}")
        effective_bs = batch_size * gacc_steps * accelerator.num_processes
        print(f"effective_batch_size: {effective_bs}")
    
    # HF Login and repo initiation
    hf_login()
    hf_api = HfApi()
    # Create the HF repo to push the model to the HF hub
    HF_REPO_ID = f"{hf_user_name}/{ckpt_name}"
    create_repo(repo_id=HF_REPO_ID, exist_ok=True)
    if accelerator.is_main_process:
        print(f"HF user: {hf_api.whoami()['name']}")
        print(f"HF repo: {HF_REPO_ID}")

    # Load and Prep data
    with accelerator.main_process_first():
        imdb_dataset = load_dataset(**dataset_dict)
        tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
        mlm_datasets = imdb_dataset\
            .train_test_split(
                train_size=train_size,
                test_size=test_size, seed=42
            )\
            .map(
                tokenize_function,
                batched=True,
                remove_columns=["text", "label"],
                fn_kwargs = {"tokenizer": tokenizer}
            )\
            .map(
                group_texts, batched=True
            )
        
    train_dataloader, eval_dataloader = create_dls(mlm_datasets, tokenizer)
    
    if accelerator.is_main_process:
        print(f"final input datasets features: {mlm_datasets["train"].features}")
        print(f"train DS length: {len(mlm_datasets["train"])}")
        print(f"eval DS length: {len(mlm_datasets["test"])}")
        print(f"train DL length: {len(train_dataloader)}")
        print(f"eval DL length: {len(eval_dataloader)}")

    # Preparing training components
    model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)
    optimizer = AdamW(model.parameters(), lr=lr)
    model, optimizer, train_dataloader, eval_dataloader = accelerator.prepare(
        model, optimizer, train_dataloader, eval_dataloader
    )
    num_update_steps_per_epoch = math.ceil(len(train_dataloader) / gacc_steps)
    num_training_steps = num_train_epochs * num_update_steps_per_epoch
    logging_steps = eval_steps = max(1, num_update_steps_per_epoch // 4)
    
    lr_scheduler = get_scheduler(
        lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=0,
        num_training_steps=num_training_steps,
    )

    accelerator.wait_for_everyone()
    rank = accelerator.process_index
    is_main = accelerator.is_main_process
    pid = f"[rank={rank} main={is_main}]"
    accelerator.print(f"{pid}|train_len={len(train_dataloader)}|eval_len={len(eval_dataloader)}")
    accelerator.print(f"{pid}|eval_steps={eval_steps}|train_opt_steps={num_training_steps}")
    accelerator.wait_for_everyone()

    # Training loop
    global_step = 0
    running_loss = 0.0
    running_mb = 0
    scheduler_steps = 0
    progress_bar = tqdm(
        total=num_training_steps,
        desc="Optimizer Updates",
        disable=not accelerator.is_main_process,
        leave=False,
    )

    for epoch in range(num_train_epochs):
        model.train()
        for step, batch in enumerate(train_dataloader):
            with accelerator.accumulate(model):
                with accelerator.autocast():
                    outputs = model(**batch)
                    loss = outputs.loss
                running_loss += loss.detach().float().item()
                running_mb += 1
                accelerator.backward(loss)
                if accelerator.sync_gradients:
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
                    lr_scheduler.step()
                    scheduler_steps += 1
            
            # We count "global_step" in terms of optimizer updates
            if accelerator.sync_gradients:
                global_step += 1
                progress_bar.update(1)

                # TRAIN logging (main process only)
                if global_step % logging_steps == 0:
                    avg_train_loss = running_loss / max(1, running_mb)
                    lr_val = optimizer.param_groups[0]["lr"]
                    # gather across ranks -> tensors of shape [num_processes]
                    loss_t = torch.tensor([avg_train_loss], device=accelerator.device)
                    lr_t   = torch.tensor([lr_val], device=accelerator.device)
                    # TODO: what happens if we use gather_for_metrics here??
                    loss_all = accelerator.gather(loss_t).detach().cpu().tolist()
                    lr_all   = accelerator.gather(lr_t).detach().cpu().tolist()
                    if accelerator.is_main_process:
                        for r, (l, lr_) in enumerate(zip(loss_all, lr_all)):
                            accelerator.print(
                                f"train_loss/rank_{r} = {l} | learning_rate/rank_{r} = {lr_}"
                            )
                    running_loss = 0.0
                    running_mb = 0
                # EVAL logging
                if (global_step % eval_steps == 0) or (global_step == num_training_steps):
                    eval_logs = run_eval2(model, eval_dataloader, accelerator)
                    if accelerator.is_main_process: print(f"Step {global_step} eval:", eval_logs)
    if accelerator.is_main_process:
        print("Scheduler steps:", scheduler_steps, "Expected:", num_training_steps)
    accelerator.wait_for_everyone()
    progress_bar.close()
    accelerator.end_training() # Close trackers
    accelerator.free_memory() # Free up memory

In [ ]:
! accelerate launch --num_processes 2 main.py